In [1]:
import os

In [8]:
%pwd

'c:\\Users\\Dell User\\Desktop\\DeepLearm\\Deep_Learning'

In [3]:
os.chdir('../')

In [4]:
from pathlib import Path
from dataclasses import dataclass

@dataclass
class BaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path:Path
    params_image_size:list
    params_learning_rate:float
    params_include_top:bool
    params_weights:str
    params_classes:int


In [5]:
# configuration manager
from src.constants import *
from src.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_base_model_config(self) -> BaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        base_model_config = BaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return base_model_config

In [6]:
import os 
import tensorflow as tf 
from zipfile import ZipFile
import urllib.request as request

class BaseModel:
    def __init__(self, config: BaseModelConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.DenseNet121(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        ) 
    
    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)

     

        x = tf.keras.layers.Dense(units=256, activation="relu")(flatten_in)
        x = tf.keras.layers.Dropout(0.5)(x)
        x = tf.keras.layers.Dense(units=128, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.3)(x)

        prediction = tf.keras.layers.Dense(units=classes,activation="softmax")(x)


        full_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=50,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )
    @staticmethod
    def save_model(path:Path,model:tf.keras.Model):
        model.save(path)

In [7]:
try:
    config = ConfigurationManager()
    prepare_base_model=config.get_base_model_config()
    base_model=BaseModel(config=prepare_base_model)
    base_model.get_base_model()
    base_model.update_base_model()
except  Exception as e:
        raise e

[2026-06-28 08:07:38,230: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 08:07:38,233: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 08:07:38,234: INFO: common: created directory at: artifacts]
[2026-06-28 08:07:38,235: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-06-28 08:07:42,410: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                